# Laboratorio — Clase 7: Detección de Objetos con YOLO
**Redes Neuronales y Aprendizaje Profundo · UNI FIIS · Unidad de Posgrado**
Dr. Aldo Camargo Fernández-Baca · Octubre 2026

<a href="https://colab.research.google.com/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

Este laboratorio tiene dos partes:

| Parte | Contenido | Hardware |
|---|---|---|
| **A. Conceptos de YOLO v1 desde cero** | IoU · codificación en rejilla $S\times S$ · arquitectura v1 · función de pérdida · decodificación + NMS · mAP · entrenamiento de un *mini-YOLO v1* | CPU o GPU |
| **B. Aplicación práctica con Roboflow** | Dataset *football-players-detection* · exploración · inferencia *zero-shot* · *fine-tuning* de **YOLO26** · evaluación · exportación | **GPU (T4)** |

> **Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)`.
> Para la Parte B necesitan una **API key gratuita de Roboflow** (https://app.roboflow.com → *Settings → API Keys*). Guárdenla en los *Secrets* de Colab (ícono 🔑) con el nombre `ROBOFLOW_API_KEY`.

**Referencias:** Redmon et al. (2016), *You Only Look Once: Unified, Real-Time Object Detection*, CVPR · Ultralytics YOLO26 (2026).

In [ ]:
import math, random, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image, ImageDraw

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

# Parte A — YOLO v1 desde cero

## A1. Intersección sobre Unión (IoU)

$$\mathrm{IoU}(A,B)=\frac{|A\cap B|}{|A|+|B|-|A\cap B|}$$

Usaremos dos formatos de caja:
* **xyxy**: esquinas $(x_1,y_1,x_2,y_2)$ — cómodo para IoU y para dibujar.
* **cxcywh**: centro y tamaño $(c_x,c_y,w,h)$ — lo que predice YOLO.

In [ ]:
def cxcywh_to_xyxy(b):
    cx, cy, w, h = b.unbind(-1)
    return torch.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], dim=-1)

def xyxy_to_cxcywh(b):
    x1, y1, x2, y2 = b.unbind(-1)
    return torch.stack([(x1+x2)/2, (y1+y2)/2, x2-x1, y2-y1], dim=-1)

def box_iou_elementwise(a, b, eps=1e-9):
    # IoU elemento a elemento entre cajas xyxy de igual forma (..., 4)
    x1 = torch.maximum(a[..., 0], b[..., 0]); y1 = torch.maximum(a[..., 1], b[..., 1])
    x2 = torch.minimum(a[..., 2], b[..., 2]); y2 = torch.minimum(a[..., 3], b[..., 3])
    inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
    area_a = (a[..., 2]-a[..., 0]).clamp(min=0) * (a[..., 3]-a[..., 1]).clamp(min=0)
    area_b = (b[..., 2]-b[..., 0]).clamp(min=0) * (b[..., 3]-b[..., 1]).clamp(min=0)
    return inter / (area_a + area_b - inter + eps)

# Verificación contra torchvision
A = torch.tensor([[0., 0., 2., 2.]]); B = torch.tensor([[1., 1., 3., 3.]])
print("IoU propia      :", box_iou_elementwise(A, B).item())     # 1/7 ≈ 0.1429
print("IoU torchvision :", torchvision.ops.box_iou(A, B).item())

In [ ]:
# Visualización: cómo cambia el IoU al desplazar una caja
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
gt = torch.tensor([0.3, 0.3, 0.7, 0.7])
for ax, dx in zip(axes, [0.0, 0.1, 0.2, 0.35]):
    pr = gt + torch.tensor([dx, dx/2, dx, dx/2])
    iou = box_iou_elementwise(gt, pr).item()
    for b, c, lab in [(gt, "tab:green", "real"), (pr, "tab:red", "pred")]:
        ax.add_patch(patches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1], fill=False, lw=2, ec=c, label=lab))
    ax.set_xlim(0, 1.2); ax.set_ylim(1.2, 0); ax.set_aspect("equal")
    ax.set_title(f"desplazamiento={dx:.2f}\nIoU={iou:.2f}"); ax.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

> **Pregunta:** en PASCAL VOC una detección cuenta como correcta si IoU ≥ 0.5. ¿Cuál de los cuatro paneles sería un *true positive*? ¿Y con el criterio COCO, que promedia umbrales de 0.50 a 0.95?

## A2. Un dataset sintético para experimentar

Para entender YOLO v1 sin depender de un dataset grande, generamos imágenes de $112\times112$ con **círculos, cuadrados y triángulos** ($C=3$). Las etiquetas están en formato YOLO: `clase cx cy w h`, todo normalizado a $[0,1]$.

Respetamos una restricción de YOLO v1 a propósito: **no ponemos dos centros en la misma celda** de la rejilla $7\times 7$ (¡v1 no podría representarlos!).

In [ ]:
CLASSES = ["círculo", "cuadrado", "triángulo"]
IMG, S, B, C = 112, 7, 2, len(CLASSES)

def make_sample(rng, max_objs=3):
    img = Image.new("RGB", (IMG, IMG), tuple(int(v) for v in rng.integers(0, 60, 3)))
    d = ImageDraw.Draw(img)
    boxes, labels, used_cells = [], [], set()
    for _ in range(rng.integers(1, max_objs + 1)):
        for _try in range(20):
            sz = int(rng.integers(16, 46))
            x1, y1 = int(rng.integers(0, IMG - sz)), int(rng.integers(0, IMG - sz))
            cx, cy = (x1 + sz/2) / IMG, (y1 + sz/2) / IMG
            cell = (int(cy * S), int(cx * S))
            if cell not in used_cells:
                break
        else:
            continue
        used_cells.add(cell)
        k = int(rng.integers(0, C))
        color = tuple(int(v) for v in rng.integers(120, 256, 3))
        if k == 0:   d.ellipse([x1, y1, x1+sz, y1+sz], fill=color)
        elif k == 1: d.rectangle([x1, y1, x1+sz, y1+sz], fill=color)
        else:        d.polygon([(x1+sz/2, y1), (x1, y1+sz), (x1+sz, y1+sz)], fill=color)
        boxes.append([cx, cy, sz/IMG, sz/IMG]); labels.append(k)
    x = torch.from_numpy(np.asarray(img, dtype=np.float32) / 255.).permute(2, 0, 1)
    return x, torch.tensor(boxes, dtype=torch.float32), torch.tensor(labels, dtype=torch.long)

def make_dataset(n, seed):
    rng = np.random.default_rng(seed)
    return [make_sample(rng) for _ in range(n)]

train_set = make_dataset(4000, seed=1)
val_set   = make_dataset(500,  seed=2)
print(len(train_set), "train |", len(val_set), "val")

In [ ]:
def draw(ax, x, boxes, labels, scores=None, grid=True, title=None):
    ax.imshow(x.permute(1, 2, 0).numpy()); ax.axis("off")
    if grid:
        for k in range(1, S):
            ax.axhline(k * IMG / S, color="w", lw=0.4, alpha=0.5); ax.axvline(k * IMG / S, color="w", lw=0.4, alpha=0.5)
    cols = ["tab:red", "tab:cyan", "tab:orange"]
    for n, (b, l) in enumerate(zip(boxes, labels)):
        cx, cy, w, h = (b * IMG).tolist()
        ax.add_patch(patches.Rectangle((cx - w/2, cy - h/2), w, h, fill=False, lw=2, ec=cols[int(l)]))
        ax.plot(cx, cy, "o", ms=3, color=cols[int(l)])
        txt = CLASSES[int(l)] + (f" {scores[n]:.2f}" if scores is not None else "")
        ax.text(cx - w/2, cy - h/2 - 2, txt, color=cols[int(l)], fontsize=7)
    if title: ax.set_title(title, fontsize=9)

fig, axes = plt.subplots(1, 5, figsize=(15, 3.3))
for ax, (x, b, l) in zip(axes, train_set[:5]):
    draw(ax, x, b, l)
plt.suptitle("Dataset sintético con rejilla 7×7 (punto = centro → celda responsable)"); plt.tight_layout(); plt.show()

## A3. Codificación del *ground truth* en el tensor objetivo

Para cada objeto con centro normalizado $(x_a, y_a)$:

$$j=\lfloor S x_a\rfloor,\quad i=\lfloor S y_a\rfloor,\qquad x = S x_a - j,\quad y = S y_a - i$$

El objetivo por celda es $[x,\ y,\ \sqrt{w},\ \sqrt{h},\ \mathbb{1}^{obj},\ \text{one-hot}(C)]$, es decir, un tensor $S\times S\times(5+C)$.
La red, en cambio, produce $S\times S\times(5B+C)$: $B$ cajas candidatas por celda que **compiten** para ser la responsable.

> Detalle de implementación: hacemos que la red prediga $\sqrt{w}$ y $\sqrt{h}$ directamente (en lugar de $w,h$) para que la pérdida de la Ec. (2) del paper sea un MSE simple y evitar raíces de números negativos.

In [ ]:
def encode_target(boxes, labels, S=S, C=C):
    t = torch.zeros(S, S, 5 + C)
    for (cx, cy, w, h), k in zip(boxes.tolist(), labels.tolist()):
        j, i = min(int(cx * S), S - 1), min(int(cy * S), S - 1)
        if t[i, j, 4] == 1:          # celda ocupada: limitación de YOLO v1 (1 objeto por celda)
            continue
        t[i, j, 0:4] = torch.tensor([cx * S - j, cy * S - i, math.sqrt(w), math.sqrt(h)])
        t[i, j, 4] = 1.0
        t[i, j, 5 + k] = 1.0
    return t

x0, b0, l0 = train_set[0]
t0 = encode_target(b0, l0)
print("Forma del objetivo:", tuple(t0.shape))
for i, j in (t0[..., 4] == 1).nonzero().tolist():
    print(f"celda (i={i}, j={j}) ->", [round(v, 3) for v in t0[i, j].tolist()])

In [ ]:
class ShapesDS(torch.utils.data.Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, k):
        x, b, l = self.data[k]
        return x, encode_target(b, l)

train_dl = torch.utils.data.DataLoader(ShapesDS(train_set), batch_size=64, shuffle=True)
val_dl   = torch.utils.data.DataLoader(ShapesDS(val_set),   batch_size=128)
xb, tb = next(iter(train_dl)); print(xb.shape, tb.shape)

## A4. La arquitectura original de YOLO v1

24 capas convolucionales + 2 FC, *Leaky ReLU* (0.1), reducciones $1\times1$ al estilo GoogLeNet, entrada $448\times448$, salida $7\times7\times30$. La implementamos fielmente para inspeccionar formas y número de parámetros (no la entrenaremos: necesitaría ImageNet + VOC y días de GPU).

In [ ]:
# (k, c_out, stride) para conv; "M" para maxpool. Notación tomada de la Figura 3 del paper.
YOLOV1_CFG = [
    (7, 64, 2), "M",
    (3, 192, 1), "M",
    (1, 128, 1), (3, 256, 1), (1, 256, 1), (3, 512, 1), "M",
    *[(1, 256, 1), (3, 512, 1)] * 4, (1, 512, 1), (3, 1024, 1), "M",
    *[(1, 512, 1), (3, 1024, 1)] * 2, (3, 1024, 1), (3, 1024, 2),
    (3, 1024, 1), (3, 1024, 1),
]

class YOLOv1(nn.Module):
    def __init__(self, S=7, B=2, C=20, cfg=YOLOV1_CFG):
        super().__init__()
        self.S, self.B, self.C = S, B, C
        layers, c_in = [], 3
        for item in cfg:
            if item == "M":
                layers.append(nn.MaxPool2d(2, 2))
            else:
                k, c_out, s = item
                layers += [nn.Conv2d(c_in, c_out, k, s, k // 2), nn.LeakyReLU(0.1)]
                c_in = c_out
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(1024 * S * S, 4096), nn.LeakyReLU(0.1),
            nn.Dropout(0.5), nn.Linear(4096, S * S * (5 * B + C)))   # salida lineal
    def forward(self, x):
        return self.head(self.features(x)).view(-1, self.S, self.S, 5 * self.B + self.C)

v1 = YOLOv1()
n_conv = sum(isinstance(m, nn.Conv2d) for m in v1.modules())
n_fc = sum(isinstance(m, nn.Linear) for m in v1.modules())
n_params = sum(p.numel() for p in v1.parameters())
print(f"Capas conv: {n_conv} | capas FC: {n_fc} | parámetros: {n_params/1e6:.1f} M")
with torch.no_grad():
    print("Salida para entrada 448×448:", tuple(v1(torch.zeros(1, 3, 448, 448)).shape))
fc_params = sum(p.numel() for p in v1.head.parameters())
print(f"¡El {100*fc_params/n_params:.0f}% de los parámetros está en las capas FC!")
del v1

> **Observación:** casi todos los parámetros están en la primera capa FC ($7\cdot7\cdot1024\times4096$). YOLOv2 elimina las FC y predice con convoluciones — una de las razones de su mayor velocidad. Compare con el análisis de VGGNet (Clase 4).

Para el experimento usamos un **mini-YOLO v1**: misma lógica (backbone convolucional → rejilla $7\times7$ → FC → tensor $S\times S\times(5B+C)$), pero pequeño para entrenar en minutos.

In [ ]:
class MiniYOLOv1(nn.Module):
    def __init__(self, S=S, B=B, C=C):
        super().__init__()
        self.S, self.B, self.C = S, B, C
        def blk(ci, co): return [nn.Conv2d(ci, co, 3, 1, 1), nn.BatchNorm2d(co), nn.LeakyReLU(0.1)]
        self.features = nn.Sequential(
            *blk(3, 32), nn.MaxPool2d(2),          # 56
            *blk(32, 64), nn.MaxPool2d(2),         # 28
            *blk(64, 128), nn.Conv2d(128, 64, 1), nn.LeakyReLU(0.1), *blk(64, 128), nn.MaxPool2d(2),   # 14
            *blk(128, 256), nn.MaxPool2d(2),       # 7
            *blk(256, 256))
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(256 * S * S, 512), nn.LeakyReLU(0.1), nn.Dropout(0.2),
            nn.Linear(512, S * S * (5 * B + C)))
    def forward(self, x):
        return self.head(self.features(x)).view(-1, self.S, self.S, 5 * self.B + self.C)

model = MiniYOLOv1().to(device)
print(f"Mini-YOLO v1: {sum(p.numel() for p in model.parameters())/1e6:.2f} M parámetros",
      "| salida:", tuple(model(xb[:2].to(device)).shape))

## A5. La función de pérdida de YOLO v1

$$
\begin{aligned}
\mathcal{L}=&\ \lambda_{\text{coord}}\sum_{i}\sum_{j}\mathbb{1}^{obj}_{ij}\big[(x_i-\hat x_i)^2+(y_i-\hat y_i)^2\big]
+\lambda_{\text{coord}}\sum_{i}\sum_{j}\mathbb{1}^{obj}_{ij}\big[(\sqrt{w_i}-\sqrt{\hat w_i})^2+(\sqrt{h_i}-\sqrt{\hat h_i})^2\big]\\
&+\sum_{i}\sum_{j}\mathbb{1}^{obj}_{ij}(C_i-\hat C_i)^2
+\lambda_{\text{noobj}}\sum_{i}\sum_{j}\mathbb{1}^{noobj}_{ij}(C_i-\hat C_i)^2
+\sum_{i}\mathbb{1}^{obj}_{i}\sum_{c}(p_i(c)-\hat p_i(c))^2
\end{aligned}
$$

con $\lambda_{\text{coord}}=5$, $\lambda_{\text{noobj}}=0.5$. El paso clave es elegir el **predictor responsable**: de las $B$ cajas de la celda, la de **mayor IoU** con la caja real.

In [ ]:
def cell_to_abs(boxes, S=S):
    # (N,S,S,...,4) en coordenadas de celda [x, y, sqrt(w), sqrt(h)] -> xyxy absoluto normalizado
    N = boxes.shape[0]
    ii = torch.arange(S, device=boxes.device).view(1, S, 1).expand(N, S, S)
    jj = torch.arange(S, device=boxes.device).view(1, 1, S).expand(N, S, S)
    while ii.dim() < boxes.dim() - 1:
        ii, jj = ii.unsqueeze(-1), jj.unsqueeze(-1)
    cx = (jj + boxes[..., 0]) / S
    cy = (ii + boxes[..., 1]) / S
    w, h = boxes[..., 2].clamp(min=0) ** 2, boxes[..., 3].clamp(min=0) ** 2
    return torch.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], dim=-1)

class YOLOv1Loss(nn.Module):
    def __init__(self, S=S, B=B, C=C, l_coord=5.0, l_noobj=0.5):
        super().__init__()
        self.S, self.B, self.C, self.l_coord, self.l_noobj = S, B, C, l_coord, l_noobj

    def forward(self, pred, target):
        N, B = pred.shape[0], self.B
        pboxes = pred[..., :5 * B].view(N, self.S, self.S, B, 5)      # (N,S,S,B,5)
        pcls = pred[..., 5 * B:]                                       # (N,S,S,C)
        tbox, obj, tcls = target[..., :4], target[..., 4], target[..., 5:]

        # 1) IoU de cada una de las B cajas con la caja real de la celda
        iou = box_iou_elementwise(cell_to_abs(pboxes[..., :4]),
                                  cell_to_abs(tbox).unsqueeze(3).expand(-1, -1, -1, B, -1))   # (N,S,S,B)
        # 2) predictor responsable = argmax IoU, sólo en celdas con objeto
        best = iou.argmax(dim=-1)
        resp = F.one_hot(best, B).float() * obj.unsqueeze(-1)          # 1^{obj}_{ij}
        noresp = 1.0 - resp                                            # 1^{noobj}_{ij}

        t = tbox.unsqueeze(3)                                          # (N,S,S,1,4)
        loss_xy = (resp * ((pboxes[..., 0:2] - t[..., 0:2]) ** 2).sum(-1)).sum()
        loss_wh = (resp * ((pboxes[..., 2:4] - t[..., 2:4]) ** 2).sum(-1)).sum()
        conf = pboxes[..., 4]
        loss_obj = (resp * (conf - iou.detach()) ** 2).sum()           # objetivo = IoU (paper)
        loss_noobj = (noresp * conf ** 2).sum()
        loss_cls = (obj.unsqueeze(-1) * (pcls - tcls) ** 2).sum()

        parts = {"xy": self.l_coord * loss_xy, "wh": self.l_coord * loss_wh, "obj": loss_obj,
                 "noobj": self.l_noobj * loss_noobj, "cls": loss_cls}
        return sum(parts.values()) / N, {k: v.item() / N for k, v in parts.items()}

criterion = YOLOv1Loss()

In [ ]:
# Prueba unitaria: una predicción "perfecta" debe tener pérdida 0
def perfect_prediction(target):
    p = torch.zeros(*target.shape[:3], 5 * B + C)
    p[..., 0:4] = target[..., 0:4]; p[..., 4] = target[..., 4]   # caja 1 = verdad, conf = 1 si hay objeto
    p[..., 5:9] = target[..., 0:4]                                # caja 2 = misma caja, conf = 0
    p[..., 5 * B:] = target[..., 5:]
    return p

tgt = torch.stack([encode_target(b, l) for _, b, l in train_set[:8]])
loss, parts = criterion(perfect_prediction(tgt), tgt)
print("Pérdida predicción perfecta:", round(loss.item(), 6), parts)
loss, parts = criterion(torch.randn(8, S, S, 5 * B + C), tgt)
print("Pérdida predicción aleatoria:", round(loss.item(), 3), {k: round(v, 2) for k, v in parts.items()})

> **Ejercicio rápido:** ponga `l_noobj=1.0` y vuelva a calcular la pérdida de la predicción aleatoria. ¿Qué término domina? Cuente cuántas celdas tienen objeto vs. cuántas no ($49\cdot B$ cajas por imagen). Esto es exactamente el desbalance que motiva $\lambda_{\text{noobj}}=0.5$.

## A6. Decodificación y *Non-Maximum Suppression*

En inferencia: puntaje por clase $=\Pr(\text{clase}\mid\text{obj})\cdot\text{conf}$, umbral, y NMS **por clase**.
Implementamos NMS a mano y verificamos contra `torchvision.ops.batched_nms`.

In [ ]:
def nms(boxes, scores, iou_thr=0.5):
    order = scores.argsort(descending=True)
    keep = []
    while order.numel() > 0:
        i = order[0]; keep.append(i.item())
        if order.numel() == 1: break
        ious = torchvision.ops.box_iou(boxes[i:i+1], boxes[order[1:]])[0]
        order = order[1:][ious <= iou_thr]
    return torch.tensor(keep, dtype=torch.long)

def decode(pred, conf_thr=0.2, iou_thr=0.5, use_nms=True):
    # pred: (S,S,5B+C) de UNA imagen -> boxes xyxy (normalizadas), scores, labels
    pb = pred[..., :5 * B].view(S, S, B, 5)
    xyxy = cell_to_abs(pb[..., :4].unsqueeze(0))[0].reshape(-1, 4)       # (S*S*B,4)
    conf = pb[..., 4].reshape(-1)
    cls_prob = pred[..., 5 * B:].unsqueeze(2).expand(S, S, B, C).reshape(-1, C)
    scores_all = cls_prob * conf.unsqueeze(-1)                             # Pr(clase)·IoU
    scores, labels = scores_all.max(-1)
    m = scores > conf_thr
    xyxy, scores, labels = xyxy[m].clamp(0, 1), scores[m], labels[m]
    if use_nms and len(scores):
        keep = []
        for c in labels.unique():                                          # NMS por clase
            idx = (labels == c).nonzero().squeeze(1)
            keep.append(idx[nms(xyxy[idx], scores[idx], iou_thr)])
        keep = torch.cat(keep)
        keep = keep[scores[keep].argsort(descending=True)]
        xyxy, scores, labels = xyxy[keep], scores[keep], labels[keep]
    return xyxy, scores, labels

# Verificación de NMS contra torchvision
bx = torch.rand(200, 2) * 0.8; bx = torch.cat([bx, bx + torch.rand(200, 2) * 0.2 + 0.05], 1)
sc, lb = torch.rand(200), torch.randint(0, 3, (200,))
mine = torch.cat([(lb == c).nonzero().squeeze(1)[nms(bx[lb == c], sc[lb == c])] for c in lb.unique()])
ref = torchvision.ops.batched_nms(bx, sc, lb, 0.5)
print("¿NMS propio == torchvision?", set(mine.tolist()) == set(ref.tolist()), f"({len(ref)} cajas conservadas de 200)")

## A7. Métrica: mAP@0.5 (estilo PASCAL VOC)

1. Ordenar todas las detecciones de una clase por puntaje.
2. Cada detección es TP si su IoU con una caja real **aún no emparejada** es ≥ 0.5; si no, FP.
3. Curva precisión–*recall* acumulada → AP = área bajo la curva interpolada. mAP = promedio sobre clases.

In [ ]:
def average_precision(rec, prec):
    mrec = np.concatenate([[0.], rec, [1.]]); mpre = np.concatenate([[0.], prec, [0.]])
    for k in range(len(mpre) - 2, -1, -1):
        mpre[k] = max(mpre[k], mpre[k + 1])           # envolvente monótona
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))

def evaluate_map(dets, gts, n_classes=C, iou_thr=0.5):
    # dets: lista por imagen de (xyxy, scores, labels); gts: lista por imagen de (xyxy, labels)
    aps = []
    for c in range(n_classes):
        records, n_gt, matched = [], 0, {}
        for k, (gb, gl) in enumerate(gts):
            n_gt += int((gl == c).sum()); matched[k] = torch.zeros(int((gl == c).sum()), dtype=torch.bool)
        for k, (db, ds, dl) in enumerate(dets):
            for b, s in zip(db[dl == c], ds[dl == c]):
                records.append((s.item(), k, b))
        records.sort(key=lambda r: -r[0])
        tp = np.zeros(len(records)); fp = np.zeros(len(records))
        for r, (s, k, b) in enumerate(records):
            g = gts[k][0][gts[k][1] == c]
            if len(g) == 0:
                fp[r] = 1; continue
            ious = torchvision.ops.box_iou(b[None], g)[0]
            best = int(ious.argmax())
            if ious[best] >= iou_thr and not matched[k][best]:
                tp[r] = 1; matched[k][best] = True
            else:
                fp[r] = 1
        tp, fp = np.cumsum(tp), np.cumsum(fp)
        rec = tp / max(n_gt, 1); prec = tp / np.maximum(tp + fp, 1e-9)
        aps.append(average_precision(rec, prec) if len(records) else 0.0)
    return float(np.mean(aps)), aps

@torch.no_grad()
def run_eval(model, data, conf_thr=0.05, use_nms=True):
    model.eval(); dets, gts = [], []
    for k in range(0, len(data), 128):
        chunk = data[k:k + 128]
        out = model(torch.stack([x for x, _, _ in chunk]).to(device)).cpu()
        for p, (_, b, l) in zip(out, chunk):
            dets.append(decode(p, conf_thr=conf_thr, use_nms=use_nms)); gts.append((cxcywh_to_xyxy(b), l))
    return evaluate_map(dets, gts)

print("mAP@0.5 del modelo SIN entrenar:", round(run_eval(model, val_set)[0], 3))

## A8. Entrenamiento del mini-YOLO v1

El paper usa SGD con *momentum* 0.9, *weight decay* $5\times10^{-4}$ y *warm-up*. Aquí usamos AdamW con *warm-up* lineal + coseno para converger en pocos minutos (≈2–3 min en una T4).

In [ ]:
EPOCHS = 30 if device.type == "cuda" else 6
model = MiniYOLOv1().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=5e-4)
steps = EPOCHS * len(train_dl); warm = len(train_dl)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / (steps - warm))))

hist = {"loss": [], "map": []}
t0 = time.time()
for ep in range(EPOCHS):
    model.train(); tot, parts_acc = 0.0, {}
    for x, t in train_dl:
        x, t = x.to(device), t.to(device)
        # aumentación simple: volteo horizontal (hay que voltear también el objetivo)
        if random.random() < 0.5:
            x = x.flip(-1); t = t.flip(2).clone()
            t[..., 0] = torch.where(t[..., 4] > 0, 1 - t[..., 0], t[..., 0])
        loss, parts = criterion(model(x), t)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        tot += loss.item()
        for k, v in parts.items(): parts_acc[k] = parts_acc.get(k, 0) + v
    hist["loss"].append(tot / len(train_dl))
    if ep % 5 == 4 or ep == EPOCHS - 1:
        m, _ = run_eval(model, val_set); hist["map"].append((ep + 1, m))
        print(f"época {ep+1:2d} | pérdida {hist['loss'][-1]:.3f} | "
              + " ".join(f"{k}={v/len(train_dl):.2f}" for k, v in parts_acc.items())
              + f" | mAP@0.5 val = {m:.3f} | {time.time()-t0:.0f}s")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].plot(range(1, len(hist["loss"]) + 1), hist["loss"]); ax[0].set_title("Pérdida YOLO v1 (train)"); ax[0].set_xlabel("época")
e, m = zip(*hist["map"]); ax[1].plot(e, m, "o-"); ax[1].set_title("mAP@0.5 (val)"); ax[1].set_xlabel("época"); ax[1].set_ylim(0, 1)
plt.tight_layout(); plt.show()

mAP, aps = run_eval(model, val_set)
mAP_nonms, _ = run_eval(model, val_set, use_nms=False)
print("AP por clase:", {c: round(a, 3) for c, a in zip(CLASSES, aps)})
print(f"mAP@0.5 con NMS = {mAP:.3f} | sin NMS = {mAP_nonms:.3f}  → efecto de NMS = {mAP - mAP_nonms:+.3f}")

In [ ]:
# Predicciones: antes y después de NMS
model.eval()
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
with torch.no_grad():
    for col, (x, b, l) in enumerate(val_set[:5]):
        p = model(x[None].to(device))[0].cpu()
        for row, use in enumerate([False, True]):
            bx, sc, lb = decode(p, conf_thr=0.15, use_nms=use)
            draw(axes[row, col], x, xyxy_to_cxcywh(bx), lb, sc.tolist(), grid=False,
                 title=("sin NMS" if not use else "con NMS") + f" ({len(sc)} cajas)")
plt.tight_layout(); plt.show()

### Preguntas de la Parte A
1. ¿Cuántas cajas produce el modelo antes del umbral? ¿Por qué NMS sólo aporta unos pocos puntos de mAP en YOLO v1 (el paper reporta 2–3)?
2. Modifique `make_sample` para permitir **dos objetos en la misma celda** (quite la restricción `used_cells`). Reentrene: ¿qué le pasa al *recall*? Relaciónelo con la limitación principal de YOLO v1.
3. Cambie $\lambda_{\text{coord}}$ a 1 y reentrene. ¿Qué parte de la pérdida empeora y cómo afecta al mAP?

---
# Parte B — Aplicación práctica: YOLO26 + Roboflow

Ahora pasamos del YOLO v1 didáctico al **estado del arte en producción**: **YOLO26** (Ultralytics, enero 2026) — *anchor-free*, multiescala y **sin NMS** (*end-to-end*).

**Dataset:** [football-players-detection](https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc) de Roboflow Universe — imágenes de partidos con 4 clases: `ball`, `goalkeeper`, `player`, `referee`. Es un dataset pequeño (~370 imágenes) y difícil: muchos jugadores pequeños y un **balón diminuto**. ¡El caso perfecto para recordar las limitaciones de v1!

In [ ]:
!pip install -q -U ultralytics roboflow
import ultralytics
ultralytics.checks()

In [ ]:
import os
from getpass import getpass
try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY") or getpass("Roboflow API key: ")

In [ ]:
from pathlib import Path
from roboflow import Roboflow

HOME = Path.cwd()
DATASET_VERSION = 1      # ver otras versiones en Roboflow Universe (pestaña "Dataset")

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
dataset = project.version(DATASET_VERSION).download("yolov8", location=str(HOME / "datasets" / "football"))
DATA_DIR = Path(dataset.location)
print(DATA_DIR); print(sorted(p.name for p in DATA_DIR.iterdir()))

### B1. Formato YOLO y el archivo `data.yaml`

Cada imagen `xxx.jpg` tiene un `xxx.txt` con una línea por objeto: `clase cx cy w h` (normalizado) — **la misma parametrización de YOLO v1** salvo que $(c_x,c_y)$ es relativo a la imagen, no a la celda.

Roboflow exporta rutas relativas que a veces no coinciden con la estructura de carpetas; las reescribimos como absolutas.

In [ ]:
import yaml
yaml_path = DATA_DIR / "data.yaml"
cfg = yaml.safe_load(open(yaml_path))
for split in ["train", "val", "test"]:
    folder = "valid" if split == "val" else split
    if (DATA_DIR / folder / "images").exists():
        cfg[split] = str(DATA_DIR / folder / "images")
cfg["path"] = str(DATA_DIR)
yaml.safe_dump(cfg, open(yaml_path, "w"), sort_keys=False)
NAMES = cfg["names"] if isinstance(cfg["names"], list) else [cfg["names"][k] for k in sorted(cfg["names"])]
print(open(yaml_path).read())

### B2. Exploración del dataset

In [ ]:
import pandas as pd
rows = []
for split in ["train", "valid", "test"]:
    for f in (DATA_DIR / split / "labels").glob("*.txt"):
        for line in open(f).read().strip().splitlines():
            c, cx, cy, w, h = line.split()[:5]
            rows.append(dict(split=split, cls=NAMES[int(c)], w=float(w), h=float(h)))
lab = pd.DataFrame(rows)
n_imgs = {s: len(list((DATA_DIR / s / "images").glob("*"))) for s in ["train", "valid", "test"]}
print("Imágenes por split:", n_imgs)
display(lab.groupby(["split", "cls"]).size().unstack(fill_value=0))

lab["area_%"] = 100 * lab.w * lab.h
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
lab[lab.split == "train"].cls.value_counts().plot.bar(ax=ax[0], title="Instancias por clase (train)")
for c in NAMES:
    ax[1].hist(lab[lab.cls == c]["area_%"], bins=np.logspace(-3, 1, 40), alpha=0.6, label=c)
ax[1].set_xscale("log"); ax[1].set_xlabel("área de la caja (% de la imagen)"); ax[1].legend(); ax[1].set_title("Tamaño de los objetos")
plt.tight_layout(); plt.show()
print(lab.groupby("cls")["area_%"].median().round(4).rename("área mediana (% imagen)"))

> **Conexión con YOLO v1:** con una rejilla $7\times7$, cada celda cubre ≈2% de la imagen. ¿Cuántos jugadores caen típicamente en la misma celda? ¿Qué tan pequeño es el balón comparado con una celda? (Mire el área mediana.)

In [ ]:
def show_yolo_samples(img_dir, n=4, names=NAMES):
    imgs = sorted(Path(img_dir).glob("*.jpg"))[:n]
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 3.4))
    cols = {"ball": "yellow", "goalkeeper": "magenta", "player": "cyan", "referee": "orange"}
    for ax, p in zip(axes, imgs):
        im = Image.open(p); W, H = im.size; ax.imshow(im); ax.axis("off")
        lbl = p.parent.parent / "labels" / (p.stem + ".txt")
        for line in open(lbl).read().strip().splitlines():
            c, cx, cy, w, h = map(float, line.split()[:5]); name = names[int(c)]
            ax.add_patch(patches.Rectangle(((cx - w/2) * W, (cy - h/2) * H), w * W, h * H,
                                           fill=False, lw=1.2, ec=cols.get(name, "red")))
    plt.tight_layout(); plt.show()

show_yolo_samples(DATA_DIR / "train" / "images")

### B3. Inferencia *zero-shot* con YOLO26 preentrenado en COCO

COCO ya contiene `person` y `sports ball`. Veamos qué hace el modelo **sin** entrenar en nuestro dataset.

In [ ]:
from ultralytics import YOLO
coco_model = YOLO("yolo26n.pt")
sample_imgs = sorted((DATA_DIR / "test" / "images").glob("*.jpg"))[:3]
res = coco_model.predict(sample_imgs, conf=0.25, verbose=False)
fig, axes = plt.subplots(1, len(res), figsize=(18, 4))
for ax, r in zip(axes, res):
    ax.imshow(r.plot()[..., ::-1]); ax.axis("off")
    ax.set_title(", ".join(f"{coco_model.names[int(k)]}×{int((r.boxes.cls == k).sum())}" for k in r.boxes.cls.unique()))
plt.suptitle("YOLO26n (COCO) sin fine-tuning"); plt.tight_layout(); plt.show()

> **Pregunta:** el modelo COCO detecta personas, pero ¿puede distinguir *jugador* vs *árbitro* vs *portero*? Esto motiva el **fine-tuning** (*transfer learning*), igual que en la Clase 3 con AlexNet preentrenado.

### B4. *Fine-tuning* de YOLO26

Parámetros clave:
* `epochs`: con ~300 imágenes de train, 50 épocas toma ≈5–8 min en una T4.
* `imgsz`: 640 por defecto. Los objetos pequeños (balón) se benefician de **1280** (≈4× más cómputo) — ver Tarea.
* El optimizador por defecto (`optimizer="auto"`) selecciona MuSGD/AdamW según el tamaño del entrenamiento.

In [ ]:
EPOCHS_FT = 50
IMGSZ = 640
model26 = YOLO("yolo26n.pt")
train_res = model26.train(data=str(yaml_path), epochs=EPOCHS_FT, imgsz=IMGSZ, batch=16,
                          project=str(HOME / "runs"), name="yolo26n_football", exist_ok=True,
                          plots=True, seed=SEED)
RUN_DIR = Path(model26.trainer.save_dir); print("Resultados en:", RUN_DIR)

In [ ]:
from IPython.display import Image as IPImage, display
display(IPImage(filename=str(RUN_DIR / "results.png"), width=1000))

In [ ]:
for name in ["confusion_matrix_normalized.png", "BoxPR_curve.png", "PR_curve.png"]:
    if (RUN_DIR / name).exists():
        display(IPImage(filename=str(RUN_DIR / name), width=650))

### B5. Evaluación en el conjunto de **test**

In [ ]:
best = YOLO(str(RUN_DIR / "weights" / "best.pt"))
metrics = best.val(data=str(yaml_path), split="test", imgsz=IMGSZ, verbose=False)
per_class = pd.DataFrame({
    "clase": [best.names[int(c)] for c in metrics.box.ap_class_index],
    "AP@0.5": metrics.box.ap50,
    "AP@[.5:.95]": metrics.box.ap,
}).round(3)
display(per_class)
print(f"mAP@0.5 = {metrics.box.map50:.3f} | mAP@[.5:.95] = {metrics.box.map:.3f} | "
      f"precisión = {metrics.box.mp:.3f} | recall = {metrics.box.mr:.3f}")

> **Discusión (Ejercicio 2 de las notas):** ¿qué clase tiene el AP más bajo? ¿Coincide con su predicción? ¿Por qué `goalkeeper` también suele ser difícil? (Pista: cuente las instancias por clase en B2.)

In [ ]:
test_imgs = sorted((DATA_DIR / "test" / "images").glob("*.jpg"))[:4]
res = best.predict(test_imgs, conf=0.3, verbose=False)
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for ax, r in zip(axes.ravel(), res):
    ax.imshow(r.plot(line_width=1, font_size=8)[..., ::-1]); ax.axis("off")
plt.suptitle("YOLO26n fine-tuned — predicciones en test"); plt.tight_layout(); plt.show()

r = res[0]
print("Formato de salida de YOLO26 (ya sin NMS externo):")
print("  boxes.xyxy :", tuple(r.boxes.xyxy.shape), "| boxes.conf:", tuple(r.boxes.conf.shape), "| boxes.cls:", tuple(r.boxes.cls.shape))
print("  tiempos (ms):", {k: round(v, 1) for k, v in r.speed.items()})

### B6. Velocidad y exportación para despliegue

YOLO26 fue diseñado para *edge*: sin NMS ni DFL, la exportación a ONNX/TensorRT/TFLite es directa.

In [ ]:
def bench(m, imgs, n=30):
    m.predict(imgs[:1], verbose=False)                  # calentamiento
    t = time.time()
    for k in range(n):
        m.predict(imgs[k % len(imgs)], verbose=False)
    return 1000 * (time.time() - t) / n

all_test = sorted((DATA_DIR / "test" / "images").glob("*.jpg"))
print(f"YOLO26n (PyTorch, {device}): {bench(best, all_test):.1f} ms/imagen (incluye pre y post-proceso)")

onnx_path = best.export(format="onnx", imgsz=IMGSZ)
onnx_model = YOLO(onnx_path, task="detect")
print(f"YOLO26n (ONNX Runtime): {bench(onnx_model, all_test):.1f} ms/imagen")

### B7. (Opcional) Comparación con YOLO11n

Active `RUN_COMPARISON = True` para entrenar YOLO11n con exactamente la misma configuración y comparar. YOLO11 todavía usa NMS y DFL.

In [ ]:
RUN_COMPARISON = False
if RUN_COMPARISON:
    m11 = YOLO("yolo11n.pt")
    m11.train(data=str(yaml_path), epochs=EPOCHS_FT, imgsz=IMGSZ, batch=16,
              project=str(HOME / "runs"), name="yolo11n_football", exist_ok=True, seed=SEED)
    best11 = YOLO(str(Path(m11.trainer.save_dir) / "weights" / "best.pt"))
    met11 = best11.val(data=str(yaml_path), split="test", imgsz=IMGSZ, verbose=False)
    display(pd.DataFrame({
        "modelo": ["YOLO26n", "YOLO11n"],
        "mAP@0.5": [metrics.box.map50, met11.box.map50],
        "mAP@[.5:.95]": [metrics.box.map, met11.box.map],
        "ms/imagen": [bench(best, all_test), bench(best11, all_test)],
    }).round(3))

---
## Tarea (entregar el notebook ejecutado)

1. **YOLO v1 (Parte A).** Implemente la variante de la pérdida que usa objetivo de confianza $C_i = 1$ en lugar de $C_i=\mathrm{IoU}$. Entrene ambas versiones con la misma semilla y compare mAP@0.5. Explique la diferencia en no más de un párrafo.
2. **Resolución y objetos pequeños (Parte B).** Reentrene YOLO26n con `imgsz=1280`. Reporte en una tabla el AP@0.5 de la clase `ball` y el tiempo por imagen para 640 vs 1280. ¿Vale la pena el costo?
3. **Tamaño del modelo.** Entrene `yolo26s.pt` (640). Construya una tabla mAP@[.5:.95] vs. latencia para n y s, al estilo de la tabla de la Sección 10 de las notas.
4. **Su propio dataset.** Busque en [Roboflow Universe](https://universe.roboflow.com) un dataset de detección relacionado con su área (industria, salud, agricultura, tránsito en Lima, etc.), entrene YOLO26n y reporte mAP@0.5, mAP@[.5:.95] y 4 imágenes de test con predicciones.
5. **Reflexión (½ página).** YOLO v1 eliminó las propuestas de regiones; YOLO26 eliminó NMS. ¿Qué componente diseñado a mano cree que desaparecerá después? Relacione con DETR y la Clase 14 (Transformers).

*Fecha de entrega: según el cronograma del curso.*